# ✨ 手勢石頭-剪刀-布 (0,2,5) TensorFlow Lite Notebook
這份 Colab 筆記示範：下載資料集 → 僅保留 0‧2‧5 → 訓練輕量 MobileNetV2 → 匯出 `rps.tflite`。Run‑All 約 2 分鐘完成。

## ⚙️ 0. 安裝 Kaggle 並下載資料集

In [ ]:
!pip install kaggle --quiet
!kaggle datasets download -d ardamavi/sign-language-digits-dataset
!unzip -q sign-language-digits-dataset.zip

## 🗂️ 1. 只保留 0‧2‧5 的影像並標籤化

In [ ]:
import pathlib, cv2, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

KEEP_DIGITS = [0, 2, 5]          # 0=石頭, 2=剪刀, 5=布
LABEL_MAP   = {0:0, 2:1, 5:2}    # 重新編碼為 0,1,2

X, y = [], []
for d in KEEP_DIGITS:
    for p in pathlib.Path(f'Dataset/{d}').glob('*.jpg'):
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (96, 96)) / 255.0
        X.append(img); y.append(LABEL_MAP[d])

X = np.array(X, dtype='float32')
y = to_categorical(y, num_classes=3)

## 🏗️ 2. 建立 MobileNetV2‑Tiny 分類器（3 類別）

In [ ]:
from tensorflow.keras import layers, models

base = tf.keras.applications.MobileNetV2(
    input_shape=(96,96,3),
    include_top=False,
    weights='imagenet')
base.trainable = False  # 冷凍預訓練權重

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.1, stratify=y)

model.fit(X_tr, y_tr,
          epochs=8, batch_size=32,
          validation_data=(X_te, y_te))

## 💾 3. 轉成 TensorFlow Lite 並下載

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
with open('rps.tflite', 'wb') as f:
    f.write(tflite_bytes)

print('✅ rps.tflite 已生成！左側檔案樹 → 右鍵 Download')